In [21]:
import os
import numpy as np
import pandas as pd
import joblib


In [23]:
import json
import hashlib
import platform
from datetime import datetime, timezone

In [24]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [25]:


if os.path.exists("/home/patel-krish/anaconda3/BrainyBeam/heart.csv"):
    df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")
elif os.path.exists("/home/patel-krish/anaconda3/BrainyBeam/heart.csv"):
    df = pd.read_csv("/home/patel-krish/anaconda3/BrainyBeam/heart.csv")
else:
    raise FileNotFoundError(
        "Place heart(2).csv or heart.csv in the notebook's working directory."
    )

df.columns = df.columns.str.strip()

if "target" not in df.columns:
    raise ValueError(
        f"'target' column not found. Available columns: {list(df.columns)}"
    )

df = df.drop_duplicates().copy()
df = df.dropna(subset=["target"]).copy()

X = df.drop(columns=["target"]).copy()
y = pd.to_numeric(df["target"], errors="coerce")

valid_rows = y.notna()
X = X.loc[valid_rows].copy()
y = y.loc[valid_rows].astype(int)

# Convert feature columns to numeric values.
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)

if y.nunique() != 2:
    raise ValueError("This code expects a binary target with two classes.")

if y.value_counts().min() < 2:
    raise ValueError("Each target class needs at least two rows for stratified splitting.")



In [26]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)



In [27]:


trained_models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Decision Tree": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "K-Nearest Neighbors": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "Support Vector Machine": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVC(
            kernel="rbf",
            class_weight="balanced"
        ))
    ])
}

for name, model in trained_models.items():
    model.fit(X_train, y_train)
    print(name, "trained successfully")



Logistic Regression trained successfully
Decision Tree trained successfully
Random Forest trained successfully
K-Nearest Neighbors trained successfully
Support Vector Machine trained successfully


In [28]:

model_results = []

for name, model in trained_models.items():
    predictions = model.predict(X_test)

    model_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions, zero_division=0
        )
    })

results = pd.DataFrame(model_results)
results = results.sort_values("F1", ascending=False).reset_index(drop=True)

print("\nMODEL COMPARISON")
print(results.to_string(index=False))

results.to_csv("all_model_performance_summary.csv", index=False)




MODEL COMPARISON
                 Model  Accuracy  Precision   Recall       F1
   K-Nearest Neighbors  0.786885   0.777778 0.848485 0.811594
   Logistic Regression  0.786885   0.794118 0.818182 0.805970
         Random Forest  0.770492   0.787879 0.787879 0.787879
         Decision Tree  0.770492   0.806452 0.757576 0.781250
Support Vector Machine  0.754098   0.764706 0.787879 0.776119


In [29]:


best_model_name = results.loc[0, "Model"]
best_model = trained_models[best_model_name]

print("\nSelected model:", best_model_name)


Selected model: K-Nearest Neighbors


In [30]:


MODEL_VERSION = "1.0.0"

safe_name = "".join(
    char if char.isalnum() or char in "-_" else "_"
    for char in best_model_name.lower().replace(" ", "_")
)

export_dir = os.path.join(
    "model_exports",
    safe_name,
    "v" + MODEL_VERSION
)

os.makedirs(export_dir, exist_ok=True)

model_path = os.path.join(
    export_dir,
    safe_name + "_v" + MODEL_VERSION + ".joblib"
)

# Compressed export using joblib
joblib.dump(
    best_model,
    model_path,
    compress=("gzip", 3),
    protocol=5
)

['model_exports/k-nearest_neighbors/v1.0.0/k-nearest_neighbors_v1.0.0.joblib']

In [31]:


def calculate_sha256(file_path):
    sha = hashlib.sha256()

    with open(file_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            sha.update(chunk)

    return sha.hexdigest()

checksum = calculate_sha256(model_path)

In [32]:


manifest = {
    "model_name": best_model_name,
    "model_version": MODEL_VERSION,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "model_file": os.path.basename(model_path),
    "features": list(X.columns),
    "target_column": "target",
    "selection_metric": "F1",
    "test_metrics": {
        key: float(results.loc[0, key])
        for key in ["Accuracy", "Precision", "Recall", "F1"]
    },
    "python_version": platform.python_version(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "scikit_learn_version": sklearn.__version__,
    "joblib_version": joblib.__version__,
    "compression": "gzip level 3",
    "pickle_protocol": 5,
    "file_size_bytes": os.path.getsize(model_path),
    "sha256": checksum
}

manifest_path = os.path.join(export_dir, "manifest.json")

with open(manifest_path, "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=4)



In [33]:


if calculate_sha256(model_path) != manifest["sha256"]:
    raise RuntimeError("Checksum verification failed.")

loaded_model = joblib.load(model_path)

test_predictions = loaded_model.predict(X_test)

print("\nEXPORT SUCCESSFUL")
print("Best model:", best_model_name)
print("Version:", MODEL_VERSION)
print("Model path:", os.path.abspath(model_path))
print("Manifest path:", os.path.abspath(manifest_path))
print("File size:", os.path.getsize(model_path), "bytes")
print("Checksum verified:", True)
print("Reload test passed:", len(test_predictions) == len(y_test))
print("\nPerformance summary saved to all_model_performance_summary.csv")



EXPORT SUCCESSFUL
Best model: K-Nearest Neighbors
Version: 1.0.0
Model path: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/model_exports/k-nearest_neighbors/v1.0.0/k-nearest_neighbors_v1.0.0.joblib
Manifest path: /home/patel-krish/anaconda3/BrainyBeam/Heart_Diesease/model_exports/k-nearest_neighbors/v1.0.0/manifest.json
File size: 10553 bytes
Checksum verified: True
Reload test passed: True

Performance summary saved to all_model_performance_summary.csv
